# Lucknow LSTM - 30-minute visibility forecast

This notebook follows the multi-step, gap-aware LSTM workflow in `guwahati_lstm_cloud_30minutes.ipynb`, adapted to Lucknow's unified LSTM-ready dataset. It forecasts `CLOUD_Vertical_Visibility` and `VIS_Visibility_MOR` for 120 steps (30 minutes) from a 240-step (one-hour) history. Rows are sampled every 15 seconds; chronological 70/15/15 splits, training-only scaling, masked targets, and gap checks are used throughout.

The notebook expects `lucknow_worked/07_LSTM_READY_DATA/lucknow_unified_lstm_dataset_aug2024_dec2025.csv` under the repository root. Checkpoints and metrics are written under `result/lucknow_lstm_cloud_30minutes/`.

In [1]:
from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PROJECT_ROOT = next(
    (path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (path / "lucknow_worked" / "07_LSTM_READY_DATA").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Could not find the repository's lucknow_worked/07_LSTM_READY_DATA directory.")

DATA_FILE = (
    PROJECT_ROOT / "lucknow_worked" / "07_LSTM_READY_DATA"
    / "lucknow_unified_lstm_dataset_aug2024_dec2025.csv"
)
ARTIFACT_DIR = PROJECT_ROOT / "result" / "lucknow_lstm_cloud_30minutes"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

TIMESTAMP_COLUMN = "Datetime"
TARGET_COLUMNS = ["CLOUD_Vertical_Visibility", "VIS_Visibility_MOR"]
LOOKBACK = 240
HORIZON = 120
SAMPLE_INTERVAL = pd.Timedelta(seconds=15)
TRAIN_RATIO = 0.70
VALIDATION_RATIO = 0.15
BATCH_SIZE = 256
EPOCHS = 50
PATIENCE = 8

if not DATA_FILE.is_file():
    raise FileNotFoundError(f"Lucknow dataset not found: {DATA_FILE}")
print("PyTorch:", torch.__version__)
print("Device:", DEVICE)
print("Data:", DATA_FILE)

PyTorch: 2.5.1
Device: cuda
Data: D:\IMD\Lucknow\lucknow_worked\07_LSTM_READY_DATA\lucknow_unified_lstm_dataset_aug2024_dec2025.csv


In [2]:
df = pd.read_csv(DATA_FILE, low_memory=False)
required_columns = [TIMESTAMP_COLUMN, *TARGET_COLUMNS]
missing_columns = sorted(set(required_columns) - set(df.columns))
if missing_columns:
    raise ValueError(f"Dataset is missing required columns: {missing_columns}")

df[TIMESTAMP_COLUMN] = pd.to_datetime(df[TIMESTAMP_COLUMN], errors="coerce")
df = (
    df.dropna(subset=[TIMESTAMP_COLUMN])
      .sort_values(TIMESTAMP_COLUMN, kind="mergesort")
      .drop_duplicates(subset=[TIMESTAMP_COLUMN], keep="first")
      .reset_index(drop=True)
)
if len(df) <= LOOKBACK + HORIZON:
    raise ValueError("The dataset is too short for the configured history and forecast horizon.")

for target in TARGET_COLUMNS:
    df[target] = pd.to_numeric(df[target], errors="coerce").replace([np.inf, -np.inf], np.nan)

predictor_columns = [
    column for column in df.columns
    if column not in [TIMESTAMP_COLUMN, *TARGET_COLUMNS]
]
df[predictor_columns] = df[predictor_columns].apply(pd.to_numeric, errors="coerce")
if "WIND_Wind_Dir" in df.columns:
    wind_direction = np.deg2rad(df["WIND_Wind_Dir"])
    df["WIND_Wind_Dir_Sin"] = np.sin(wind_direction)
    df["WIND_Wind_Dir_Cos"] = np.cos(wind_direction)
    predictor_columns.remove("WIND_Wind_Dir")
    predictor_columns.extend(["WIND_Wind_Dir_Sin", "WIND_Wind_Dir_Cos"])

timestamps = pd.DatetimeIndex(df[TIMESTAMP_COLUMN])
gaps = timestamps.to_series().diff().ne(SAMPLE_INTERVAL).to_numpy(dtype=bool)
gaps[0] = False

train_end = int(len(df) * TRAIN_RATIO)
validation_end = int(len(df) * (TRAIN_RATIO + VALIDATION_RATIO))
feature_frame = df[predictor_columns].replace([np.inf, -np.inf], np.nan)
usable_features = [
    column for column in predictor_columns
    if feature_frame.iloc[:train_end][column].notna().any()
]
dropped_features = sorted(set(predictor_columns) - set(usable_features))
if not usable_features:
    raise ValueError("No numeric predictor columns have usable training-period values.")
FEATURE_COLUMNS = usable_features
feature_frame = feature_frame[FEATURE_COLUMNS]

feature_medians = feature_frame.iloc[:train_end].median().fillna(0.0)
feature_scaler = StandardScaler()
feature_scaler.fit(feature_frame.iloc[:train_end].fillna(feature_medians))
values = feature_scaler.transform(feature_frame.fillna(feature_medians)).astype(np.float32)
if not np.isfinite(values).all():
    raise ValueError("Non-finite predictor values remain after training-only imputation and scaling.")

targets = df[TARGET_COLUMNS].to_numpy(dtype=np.float32)
target_mask = np.isfinite(targets)
target_training = targets[:train_end]
target_training_mask = target_mask[:train_end]
if not target_training_mask.any(axis=0).all():
    raise ValueError("Each target must have at least one valid observation in the training period.")
target_means = np.array([
    target_training[target_training_mask[:, index], index].mean()
    for index in range(len(TARGET_COLUMNS))
], dtype=np.float32)
target_scales = np.array([
    target_training[target_training_mask[:, index], index].std()
    for index in range(len(TARGET_COLUMNS))
], dtype=np.float32)
target_scales[target_scales == 0] = 1.0
filled_targets = np.where(target_mask, targets, target_means)
targets_scaled = ((filled_targets - target_means) / target_scales).astype(np.float32)
label_mask = target_mask.astype(np.float32)

print("Rows:", len(df), "range:", timestamps.min(), "to", timestamps.max())
print("Predictors:", len(FEATURE_COLUMNS), FEATURE_COLUMNS)
print("Dropped predictors with no training values:", dropped_features)
print("Valid target labels:", dict(zip(TARGET_COLUMNS, target_mask.sum(axis=0).tolist())))
print("15-second gaps / irregular intervals:", int(gaps.sum()))

Rows: 2520761 range: 2024-08-08 00:00:00 to 2025-12-31 23:59:45
Predictors: 20 ['CLOUD_Cloud_Base_1', 'CLOUD_Cloud_Base_2', 'CLOUD_Cloud_Base_3', 'CLOUD_OCTA1', 'CLOUD_OCTA2', 'CLOUD_OCTA3', 'CLOUD_OCTA4', 'PTU_Pressure_QNH', 'PTU_Temperature', 'PTU_DewPoint', 'PTU_Humidity', 'PW_Rain_Intensity', 'PW_Rain_Sum', 'VIS_RVR', 'WIND_Wind_Speed', 'WIND_Wind_Gust', 'Hour_Sin', 'Hour_Cos', 'WIND_Wind_Dir_Sin', 'WIND_Wind_Dir_Cos']
Dropped predictors with no training values: ['CLOUD_SITE']
Valid target labels: {'CLOUD_Vertical_Visibility': 2520761, 'VIS_Visibility_MOR': 2520761}
15-second gaps / irregular intervals: 1265


In [3]:
# Candidate endpoints are vectorized; each endpoint is the first forecast row.
candidate_ends = np.arange(LOOKBACK, len(df) - HORIZON + 1, dtype=np.int64)
starts = candidate_ends - LOOKBACK
stops = candidate_ends + HORIZON

# gaps[i] marks a non-15-second interval from row i-1 to row i.
gap_prefix = np.concatenate(([0], np.cumsum(gaps, dtype=np.int64)))
crosses_gap = (gap_prefix[stops] - gap_prefix[starts + 1]) > 0
has_future_label = target_mask.any(axis=1)
label_prefix = np.concatenate(([0], np.cumsum(has_future_label, dtype=np.int64)))
has_any_future_label = (label_prefix[stops] - label_prefix[candidate_ends]) > 0
window_ends = candidate_ends[~crosses_gap & has_any_future_label]

# Keep forecast windows away from chronological split boundaries.
train_windows = window_ends < (train_end - HORIZON)
validation_windows = (
    (window_ends >= train_end + HORIZON)
    & (window_ends < validation_end - HORIZON)
)
test_windows = window_ends >= validation_end + HORIZON
if not train_windows.any() or not validation_windows.any() or not test_windows.any():
    raise ValueError("Not enough valid gap-free windows for train/validation/test splits.")

print(
    "Gap-free windows:", len(window_ends),
    "train/validation/test:",
    int(train_windows.sum()), int(validation_windows.sum()), int(test_windows.sum()),
)

Gap-free windows: 2357129 train/validation/test: 1657846 363944 334859


In [4]:
class WindowDataset(Dataset):
    def __init__(self, ends):
        self.ends = np.asarray(ends, dtype=np.int64)

    def __len__(self):
        return len(self.ends)

    def __getitem__(self, index):
        end = self.ends[index]
        return (
            torch.from_numpy(values[end - LOOKBACK:end]),
            torch.from_numpy(targets_scaled[end:end + HORIZON]),
            torch.from_numpy(label_mask[end:end + HORIZON]),
        )

class MultiOutputLSTM(nn.Module):
    def __init__(self):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=len(FEATURE_COLUMNS), hidden_size=128,
            num_layers=2, dropout=0.1, batch_first=True,
        )
        self.head = nn.Sequential(
            nn.Linear(128, 64), nn.ReLU(),
            nn.Linear(64, HORIZON * len(TARGET_COLUMNS)),
        )

    def forward(self, inputs):
        output, _ = self.lstm(inputs)
        forecast = self.head(output[:, -1, :])
        return forecast.reshape(-1, HORIZON, len(TARGET_COLUMNS))

train_loader = DataLoader(WindowDataset(window_ends[train_windows]), batch_size=BATCH_SIZE, shuffle=False)
validation_loader = DataLoader(WindowDataset(window_ends[validation_windows]), batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(WindowDataset(window_ends[test_windows]), batch_size=BATCH_SIZE, shuffle=False)
model = MultiOutputLSTM().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss(reduction="none")

best_loss = float("inf")
best_state = None
no_improvement = 0
for epoch in range(EPOCHS):
    model.train()
    for inputs, labels, masks in train_loader:
        inputs, labels, masks = inputs.to(DEVICE), labels.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad()
        losses = criterion(model(inputs), labels)
        loss = (losses * masks).sum() / masks.sum().clamp_min(1)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

    model.eval()
    validation_losses = []
    with torch.no_grad():
        for inputs, labels, masks in validation_loader:
            inputs, labels, masks = inputs.to(DEVICE), labels.to(DEVICE), masks.to(DEVICE)
            losses = criterion(model(inputs), labels)
            validation_losses.append(((losses * masks).sum() / masks.sum().clamp_min(1)).item())
    validation_loss = float(np.mean(validation_losses))
    print(f"Epoch {epoch + 1:03d}/{EPOCHS} - val_loss: {validation_loss:.6f}")
    if validation_loss < best_loss:
        best_loss = validation_loss
        no_improvement = 0
        best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}
    else:
        no_improvement += 1
        if no_improvement >= PATIENCE:
            print("Early stopping.")
            break

if best_state is None:
    raise RuntimeError("Training did not produce a valid model checkpoint.")
model.load_state_dict(best_state)
checkpoint_path = ARTIFACT_DIR / "best_lucknow_cloud_lstm_30minutes.pt"
torch.save({
    "model_state_dict": model.state_dict(),
    "features": FEATURE_COLUMNS,
    "targets": TARGET_COLUMNS,
    "feature_medians": feature_medians.to_dict(),
    "feature_mean": feature_scaler.mean_,
    "feature_scale": feature_scaler.scale_,
    "target_means": target_means,
    "target_scales": target_scales,
    "lookback": LOOKBACK,
    "horizon": HORIZON,
}, checkpoint_path)
print("Saved checkpoint:", checkpoint_path)

Epoch 001/50 - val_loss: 0.269975
Epoch 002/50 - val_loss: 0.287156
Epoch 003/50 - val_loss: 0.265988
Epoch 004/50 - val_loss: 0.266916
Epoch 005/50 - val_loss: 0.253134
Epoch 006/50 - val_loss: 0.282111
Epoch 007/50 - val_loss: 0.252747
Epoch 008/50 - val_loss: 0.266368
Epoch 009/50 - val_loss: 0.259352
Epoch 010/50 - val_loss: 0.238287
Epoch 011/50 - val_loss: 0.256913
Epoch 012/50 - val_loss: 0.242177
Epoch 013/50 - val_loss: 0.260426
Epoch 014/50 - val_loss: 0.245775
Epoch 015/50 - val_loss: 0.257926
Epoch 016/50 - val_loss: 0.260537
Epoch 017/50 - val_loss: 0.258648
Epoch 018/50 - val_loss: 0.268468
Early stopping.
Saved checkpoint: D:\IMD\Lucknow\result\lucknow_lstm_cloud_30minutes\best_lucknow_cloud_lstm_30minutes.pt


In [ ]:
def inverse_targets(scaled_values):
    return scaled_values * target_scales.reshape(1, 1, -1) + target_means.reshape(1, 1, -1)

lead_times = [("+5 min", 19), ("+15 min", 59), ("+30 min", 119)]
overall_stats = {
    target: {"count": 0, "absolute_error": 0.0, "squared_error": 0.0,
             "actual_sum": 0.0, "actual_squared_sum": 0.0}
    for target in TARGET_COLUMNS
}
lead_parts = {
    (target, name): {"actual": [], "predicted": [], "mask": []}
    for target in TARGET_COLUMNS for name, _ in lead_times
}
model.eval()
with torch.no_grad():
    for inputs, labels, masks in test_loader:
        predictions_scaled = model(inputs.to(DEVICE)).cpu().numpy()
        labels_scaled = labels.numpy()
        masks = masks.numpy().astype(bool)
        predictions = inverse_targets(predictions_scaled)
        actual = inverse_targets(labels_scaled)
        for target_index, target in enumerate(TARGET_COLUMNS):
            valid_all = masks[:, :, target_index]
            actual_all = actual[:, :, target_index][valid_all]
            predicted_all = predictions[:, :, target_index][valid_all]
            errors = predicted_all - actual_all
            stats = overall_stats[target]
            stats["count"] += len(actual_all)
            stats["absolute_error"] += np.abs(errors).sum().item()
            stats["squared_error"] += np.square(errors).sum().item()
            stats["actual_sum"] += actual_all.sum().item()
            stats["actual_squared_sum"] += np.square(actual_all).sum().item()
            for lead_name, lead_index in lead_times:
                valid = masks[:, lead_index, target_index]
                part = lead_parts[(target, lead_name)]
                part["actual"].append(actual[valid, lead_index, target_index])
                part["predicted"].append(predictions[valid, lead_index, target_index])

rows = []
for target in TARGET_COLUMNS:
    stats = overall_stats[target]
    if stats["count"] == 0:
        raise ValueError(f"No measured test labels are available for target {target}.")
    count = stats["count"]
    total_variance = stats["actual_squared_sum"] - stats["actual_sum"] ** 2 / count
    rows.append({
        "target": target, "lead_time": "overall", "measured_samples": count,
        "MAE": stats["absolute_error"] / count,
        "RMSE": (stats["squared_error"] / count) ** 0.5,
        "R2": 1 - stats["squared_error"] / total_variance
        if count > 1 and total_variance > 0 else np.nan,
    })
    for lead_name, _ in lead_times:
        part = lead_parts[(target, lead_name)]
        actual_lead = np.concatenate(part["actual"])
        predicted_lead = np.concatenate(part["predicted"])
        rows.append({
            "target": target, "lead_time": lead_name, "measured_samples": len(actual_lead),
            "MAE": mean_absolute_error(actual_lead, predicted_lead) if len(actual_lead) else np.nan,
            "RMSE": mean_squared_error(actual_lead, predicted_lead) ** 0.5 if len(actual_lead) else np.nan,
            "R2": r2_score(actual_lead, predicted_lead) if len(actual_lead) > 1 else np.nan,
        })

metrics = pd.DataFrame(rows)
metrics_path = ARTIFACT_DIR / "test_metrics.csv"
metrics.to_csv(metrics_path, index=False)
display(metrics)
print("Saved metrics:", metrics_path)

,target,lead_time,measured_samples,MAE,RMSE,R2
0,CLOUD_Vertical_Visibility,overall,40183080,584.507785,2470.218952,0.372507
1,CLOUD_Vertical_Visibility,+5 min,334859,596.667480,2460.762788,0.374899
2,CLOUD_Vertical_Visibility,+15 min,334859,585.510498,2470.173678,0.372502
3,CLOUD_Vertical_Visibility,+30 min,334859,569.384949,2495.728150,0.362735
4,VIS_Visibility_MOR,overall,40183080,1644.986420,2130.958864,0.636954
5,VIS_Visibility_MOR,+5 min,334859,1648.153809,2135.433680,0.635215
6,VIS_Visibility_MOR,+15 min,334859,1632.615723,2118.565434,0.641199
7,VIS_Visibility_MOR,+30 min,334859,1642.141357,2126.028692,0.638838


Saved metrics: D:\IMD\Lucknow\result\lucknow_lstm_cloud_30minutes\test_metrics.csv


In [6]:
if len(df) < LOOKBACK:
    raise ValueError("Not enough rows for latest-window inference.")
if gaps[-LOOKBACK + 1:].any():
    raise ValueError("The latest history window crosses a timestamp gap; no forecast produced.")

model.eval()
inference_input = torch.from_numpy(values[-LOOKBACK:].copy()).unsqueeze(0).to(DEVICE)
with torch.no_grad():
    forecast_scaled = model(inference_input).cpu().numpy()[0]
forecast = forecast_scaled * target_scales + target_means
future_timestamps = pd.date_range(
    start=timestamps[-1] + SAMPLE_INTERVAL,
    periods=HORIZON,
    freq=SAMPLE_INTERVAL,
)
forecast_df = pd.DataFrame(forecast, index=future_timestamps, columns=TARGET_COLUMNS)
forecast_df.index.name = TIMESTAMP_COLUMN
display(forecast_df)
print("Forecast interval:", forecast_df.index.min(), "to", forecast_df.index.max())

,CLOUD_Vertical_Visibility,VIS_Visibility_MOR
Datetime,,
2026-01-01 00:00:00,14984.978516,4245.076660
2026-01-01 00:00:15,14977.065430,4243.651367
2026-01-01 00:00:30,14981.294922,4246.377441
2026-01-01 00:00:45,14985.503906,4248.399414
2026-01-01 00:01:00,14982.407227,4260.278809
...,...,...
2026-01-01 00:28:45,14984.375977,4284.861328
2026-01-01 00:29:00,14980.621094,4287.909180
2026-01-01 00:29:15,14979.834961,4283.768555


Forecast interval: 2026-01-01 00:00:00 to 2026-01-01 00:29:45
